# 🍄 EDA — Super Mario Bros Environment
**Team MOGU — Session 4 Milestone**

This notebook explores the Mario environment: action space, observation space, frame preprocessing, and reward structure.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY, SIMPLE_MOVEMENT
from models.wrappers import make_mario_env, SkipFrame, GrayScaleObservation, ResizeObservation

print('Imports OK')

## 1. Raw Environment

In [ ]:
raw_env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0', apply_api_compatibility=True, render_mode='rgb_array')
raw_env = JoypadSpace(raw_env, RIGHT_ONLY)

print(f'Action space : {raw_env.action_space}')
print(f'Actions      : {RIGHT_ONLY}')
print(f'Obs space    : {raw_env.observation_space}')
print(f'Obs shape    : {raw_env.observation_space.shape}  (H x W x C)')

In [ ]:
# Visualise 4 raw frames
obs, _ = raw_env.reset()
frames = [obs]
for _ in range(3):
    obs, _, term, trunc, _ = raw_env.step(raw_env.action_space.sample())
    frames.append(obs)
    if term or trunc:
        break

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f'Frame {i}')
    ax.axis('off')
plt.suptitle('Raw RGB Frames (240x256x3)', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Preprocessing Pipeline

In [ ]:
import cv2

raw_frame = frames[0]
gray_frame = cv2.cvtColor(raw_frame, cv2.COLOR_RGB2GRAY)
resized_frame = cv2.resize(gray_frame, (84, 84), interpolation=cv2.INTER_AREA)
normalized_frame = resized_frame / 255.0

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(raw_frame)
axes[0].set_title(f'Raw RGB\n{raw_frame.shape}')
axes[1].imshow(gray_frame, cmap='gray')
axes[1].set_title(f'Grayscale\n{gray_frame.shape}')
axes[2].imshow(resized_frame, cmap='gray')
axes[2].set_title(f'Resized (84x84)\n{resized_frame.shape}')
for ax in axes: ax.axis('off')
plt.suptitle('Preprocessing Pipeline', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Input size reduction: {raw_frame.nbytes:,} bytes → {resized_frame.nbytes:,} bytes ({resized_frame.nbytes/raw_frame.nbytes*100:.1f}%)')

## 3. Frame Stacking (Motion Information)

In [ ]:
env = make_mario_env(world=1, stage=1)
obs, _ = env.reset()
print(f'Preprocessed observation shape: {obs.shape}  (n_stack x H x W)')
print(f'Observation dtype: {obs.dtype}')
print(f'Value range: [{obs.min():.3f}, {obs.max():.3f}]')

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(obs[i], cmap='gray')
    ax.set_title(f'Stack {i}')
    ax.axis('off')
plt.suptitle('Frame Stack — 4 consecutive frames give motion cues', fontweight='bold')
plt.tight_layout()
plt.show()
env.close()

## 4. Reward Analysis — Random Policy

In [ ]:
# Run 10 random-policy episodes to understand reward baseline
env = make_mario_env(world=1, stage=1, clip_rewards=False)  # unclipped for analysis

episode_rewards = []
step_rewards = []

for ep in range(10):
    obs, _ = env.reset()
    done = False
    ep_reward = 0
    while not done:
        action = env.action_space.sample()
        obs, reward, term, trunc, info = env.step(action)
        ep_reward += reward
        step_rewards.append(reward)
        done = term or trunc
    episode_rewards.append(ep_reward)
    print(f'  Episode {ep+1}: reward={ep_reward:.1f}')

env.close()

print(f'\nRandom policy mean reward: {np.mean(episode_rewards):.1f} ± {np.std(episode_rewards):.1f}')
print(f'DQN target: 1500 | PPO target: 2500')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(episode_rewards)+1), episode_rewards, color='#4A90D9')
axes[0].axhline(np.mean(episode_rewards), color='red', linestyle='--', label=f'Mean: {np.mean(episode_rewards):.1f}')
axes[0].axhline(1500, color='orange', linestyle='--', alpha=0.7, label='DQN Target: 1500')
axes[0].axhline(2500, color='green', linestyle='--', alpha=0.7, label='PPO Target: 2500')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Episode Rewards — Random Policy')
axes[0].legend()

axes[1].hist(step_rewards, bins=30, color='#4A90D9', edgecolor='white')
axes[1].set_xlabel('Step Reward')
axes[1].set_ylabel('Count')
axes[1].set_title('Step-level Reward Distribution')

plt.tight_layout()
plt.show()

## 5. Summary

| Property | Value |
|---|---|
| Raw obs shape | (240, 256, 3) |
| Preprocessed shape | (4, 84, 84) |
| Action space | 7 discrete (RIGHT_ONLY) |
| Frame skip | 4 |
| Random policy reward | ~200-600 |
| DQN target | 1500 |
| PPO target | 2500 |